# Skin Cancer Detection: Kaggle eğitim koşusu

Bu notebook Mac'ten `kaggle kernels push -p kaggle` ile gönderilip Kaggle'da arka planda çalışır
(tarayıcı açık kalmak zorunda değil). Colab notebook'undan farkları:
- Veri Kaggle'daki HAM10000 veri setinden doğrudan bağlanır; API key gerekmez.
- Kod `/tmp`'ye klonlanır, böylece Kaggle çıktısına sadece sonuçlar (`/kaggle/working/models`) girer.
- Sonuçlar Mac'ten `kaggle kernels output yunusemreozkan/skin-cancer-train` ile çekilir.

Hangi modelin eğitileceği aşağıdaki ayar hücresinde.

In [ ]:
# Bu koşunun ayarlari: mimari degisince sadece burasi degisiyor.
ARCHITECTURE = 'convnext_tiny'
# Tarif v2 (2026-10-16): finetune lr 1e-4 + warmup + cosine, label smoothing 0.1, 25 finetune epoch.
# Cosine LR sona dogru dustugu icin early stopping fiilen kapali (patience 30); en iyi epoch val_f1 ile secilir.
EXTRA_TRAIN_ARGS = '--finetune-lr 1e-4 --scheduler warmup_cosine --warmup-epochs 2 --label-smoothing 0.1 --finetune-epochs 25 --patience 30'
# True: ISIC 2019'un HAM disi goruntuleri (SCC haric) SADECE train'e eklenir; val/test HAM10000'de kalir.
USE_ISIC2019 = False

In [ ]:
!nvidia-smi

In [ ]:
%cd /tmp
!rm -rf skin-cancer-detection
!git clone --depth 1 https://github.com/yunus345/skin-cancer-detection.git
%cd /tmp/skin-cancer-detection
!git log --oneline -1

In [ ]:
# Kaggle'da torch/torchvision CUDA'li kurulu geliyor; sadece augmentation kutuphanesini
# yerelde test ettigimiz surume sabitliyoruz.
!pip install -q albumentations==2.0.8

In [ ]:
import glob, os

# Kaggle'in veri setini bagladigi klasoru metadata dosyasindan buluyoruz (yol zamanla degisebiliyor).
matches = glob.glob('/kaggle/input/**/HAM10000_metadata.csv', recursive=True)
assert matches, 'HAM10000_metadata.csv bulunamadi: veri seti bagli mi?'
DATA_ROOT = os.path.dirname(matches[0])
print('Veri klasoru:', DATA_ROOT)

!python -m src.data.build_manifest --data-root {DATA_ROOT} --out data/processed/manifest.csv
!python -m src.data.split_manifest --manifest data/processed/manifest.csv --out data/processed/manifest.csv
!python -m src.data.validate_manifest --manifest data/processed/manifest.csv

TRAIN_MANIFEST = 'data/processed/manifest.csv'
if USE_ISIC2019:
    # Goruntuler Kaggle'daki kopyadan (andrewmvd/isic-2019); etiket ve metadata ISIC'in resmi adresinden.
    images = glob.glob('/kaggle/input/**/ISIC_2019_Training_Input/ISIC_0000000.jpg', recursive=True)
    assert images, 'ISIC 2019 goruntuleri bulunamadi: veri seti bagli mi?'
    ISIC2019_IMAGES = os.path.dirname(images[0])
    !mkdir -p data/external/isic2019
    !curl -sS -o data/external/isic2019/gt.csv https://isic-archive.s3.amazonaws.com/challenges/2019/ISIC_2019_Training_GroundTruth.csv
    !curl -sS -o data/external/isic2019/meta.csv https://isic-archive.s3.amazonaws.com/challenges/2019/ISIC_2019_Training_Metadata.csv
    !python -m src.data.add_isic2019 --ground-truth data/external/isic2019/gt.csv --metadata data/external/isic2019/meta.csv --images-dir {ISIC2019_IMAGES} --out data/processed/manifest_isic2019.csv
    # Goruntu varligi add_isic2019'da kontrol edildi; burada sizinti ve dis test kontrolu.
    !python -m src.data.validate_manifest --manifest data/processed/manifest_isic2019.csv --skip-image-check
    TRAIN_MANIFEST = 'data/processed/manifest_isic2019.csv'
print('Egitim manifest:', TRAIN_MANIFEST)

In [ ]:
# Kaggle GPU makinelerinde 4 CPU cekirdegi var; veri yuklemeyi 4 yardimci surece dagitiyoruz.
!python -m src.training.train \
  --feature-extract-epochs 5 \
  --finetune-epochs 20 \
  --patience 6 \
  --architecture {ARCHITECTURE} \
  --manifest {TRAIN_MANIFEST} \
  --num-workers 4 \
  --device auto {EXTRA_TRAIN_ARGS}

In [ ]:
!python -m src.evaluation.evaluate --device auto
!python -m src.evaluation.plot_history
!python -m src.evaluation.plot_confusion_matrix

In [ ]:
# Sadece /kaggle/working altindakiler Kaggle ciktisi olarak saklaniyor.
!rm -rf /kaggle/working/models
!cp -r models /kaggle/working/models
!ls -la /kaggle/working/models